# Lesson 01 - Вступ до AI агентів

Ласкаво просимо до першого уроку курсу **AI агенти для початківців**!

**AI агент** — це програма, яка використовує велику мовну модель (LLM) як свій механізм мислення і може здійснювати *дії* у реальному світі — викликати API, запитувати бази даних або запускати код — щоб досягти мети від імені користувача.

У цьому зошиті ви створите свого першого агента: **Туристичного агента**, який рекомендує місця відпочинку. По дорозі ви навчитеся:

1. Підключатися до Azure AI Foundry Agent Service за допомогою **Microsoft Agent Framework**.
2. Надати агенту **інструмент** — просту функцію на Python, яку він може викликати.
3. Запустити агента та проаналізувати його відповідь.
4. Потоково отримувати відповідь агента по одному токену.


## Налаштування

Перед запуском цього блокнота переконайтеся, що у вас є:

1. **Проєкт Azure AI Foundry** із розгорнутою чат-моделлю (наприклад, `gpt-4o-mini`).
2. **Вхід у систему через Azure CLI** — виконайте `az login` у вашому терміналі.
3. **Встановлені необхідні змінні оточення:**
   - `AZURE_AI_PROJECT_ENDPOINT` — кінцева точка вашого проєкту Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — назва вашої розгорнутої моделі.

Клітинка нижче встановлює необхідні пакети Python.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Створення Вашого першого агента

Агенту потрібні дві речі:

- **Інструкції**, які кажуть йому *хто він* і *як поводитися* (системний запит).
- **Інструменти** — функції Python, декоровані `@tool`, які агент може викликати для отримання інформації або виконання дій.

Нижче ми визначаємо простий інструмент, який повертає список популярних місць для відпочинку. Агент використовуватиме цей інструмент, коли користувач запитує рекомендації щодо подорожей.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Для більш інтерактивного досвіду ви можете **потоково передавати** відповідь агента. Замість того, щоб чекати повної відповіді, агент видає текстові фрагменти, щойно вони створюються. Це особливо корисно в чат-інтерфейсах, де ви хочете відображати результат у реальному часі.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Підсумок

У цьому уроці ви дізналися, як:

- **Створити провайдера**, який підключається до Azure AI Foundry Agent Service через `AzureAIProjectAgentProvider`.
- **Визначити інструмент** за допомогою декоратора `@tool`, щоб агент міг викликати ваші функції на Python.
- **Запустити агента** з повідомленням користувача та вивести його відповідь.
- **Виводити відповіді у потоці** для виводу в режимі реального часу.

У наступному уроці ми детальніше розглянемо агентські фреймворки та навчимося надавати агентам потужніші інструменти та здатність до багатокрокового логічного мислення.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу автоматичного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми докладаємо зусиль для забезпечення точності, просимо враховувати, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується звертатися до професійного людського перекладу. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
